In [ ]:
import cv2
import numpy as np
import pytesseract
from PIL import Image

def preprocess_image(image_path):
    # Read the image
    img = cv2.imread(image_path)
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Apply thresholding to preprocess the image
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    
    # Apply dilation to connect text components
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    dilation = cv2.dilate(thresh, kernel, iterations=1)
    
    return dilation, img

def find_characters(preprocessed_img):
    # Find contours for individual characters
    contours, hierarchy = cv2.findContours(preprocessed_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Sort contours from left to right
    contours = sorted(contours, key=lambda x: cv2.boundingRect(x)[0])
    
    return contours

def recognize_characters(img, contours):
    recognized_text = ""
    
    for contour in contours:
        # Get rectangle bounding contour
        x, y, w, h = cv2.boundingRect(contour)
        
        # Extract character region
        char_region = img[y:y+h, x:x+w]
        
        # Convert to PIL Image
        pil_img = Image.fromarray(char_region)
        
        # Use Tesseract to recognize single character
        char = pytesseract.image_to_string(pil_img, config='--psm 10')
        
        # Add recognized character to result
        if char.strip():  # Only add non-empty characters
            recognized_text += char.strip()
    
    return recognized_text

def handwriting_recognition(image_path):
    # Set Tesseract path if needed
    # pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
    
    # Preprocess the image
    preprocessed_img, original_img = preprocess_image(image_path)
    
    # Find individual characters
    contours = find_characters(preprocessed_img)
    
    # Recognize characters
    text = recognize_characters(preprocessed_img, contours)
    
    return text

# Example usage
if __name__ == "__main__":
    image_path = "path_to_your_handwritten_image.jpg"
    try:
        result = handwriting_recognition(image_path)
        print("Recognized Text:", result)
    except Exception as e:
        print(f"An error occurred: {str(e)}")